In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import medfilt
from scipy.ndimage import maximum_filter1d

Récupération des X vitesses angulaires

In [19]:
folder_path = "C:/Users/roman/Documents/BEaCHILD/X_et_Y" 
folder = os.listdir(folder_path)

# Initialisation des listes contenant les données des capteurs
X_gyro_dom = []
Y_gyro_dom = []
Z_gyro_dom = []
X_gyro_non_dom = []
Y_gyro_non_dom = []
Z_gyro_non_dom = []

for file in folder: 
    print(f"On est dans le fichier : {file}")

    # Extension du fichier
    extension = os.path.splitext(file)[1] 

    if extension == ".csv":
        # Récupération des données des capteurs
        data_file = pd.read_csv(folder_path + "/" + file, header = 5, names = ["Timestamp","Gyro X","Gyro Y","Gyro Z","Accelerometer X","Accelerometer Y","Accelerometer Z","Event","Quat W","Quat X","Quat Y","Quat Z","None"])    
        
        if file[-7:-4] == "dom":
            X_gyro_dom.extend(data_file["Gyro X"].values.tolist()) #.append(data_file["Gyro X"][data])
            Y_gyro_dom.extend(data_file["Gyro Y"].values.tolist()) #.append(data_file["Gyro Y"][data])
            Z_gyro_dom.extend(data_file["Gyro Z"].values.tolist()) #.append(data_file["Gyro Z"][data])

        elif file[-11:-4] == "non_dom":
            X_gyro_non_dom.extend(data_file["Gyro X"].values.tolist()) #append(data_file["Gyro X"][data])
            Y_gyro_non_dom.extend(data_file["Gyro Y"].values.tolist()) #append(data_file["Gyro Y"][data])
            Z_gyro_non_dom.extend(data_file["Gyro Z"].values.tolist()) #append(data_file["Gyro Z"][data])

print(len(X_gyro_dom))
# Vérifier que les données angulaires sont les bonnes (FAIT) puis
# Il faut faire une liste .append pour metrte toutes les données dans une liste

On est dans le fichier : Ceux_ou_ya_pas_de_Y
On est dans le fichier : Data_10_dom.csv
On est dans le fichier : Data_10_non_dom.csv
On est dans le fichier : Data_10_X.xlsx
On est dans le fichier : Data_11_dom.csv
On est dans le fichier : Data_11_non_dom.csv
On est dans le fichier : Data_11_X.xlsx
On est dans le fichier : Data_16_dom.csv
On est dans le fichier : Data_16_non_dom.csv
On est dans le fichier : Data_16_X.xlsx
On est dans le fichier : Data_19_dom.csv
On est dans le fichier : Data_19_non_dom.csv
On est dans le fichier : Data_19_X.xlsx
On est dans le fichier : Data_1_dom.csv
On est dans le fichier : Data_1_non_dom.csv
On est dans le fichier : Data_1_X.xlsx
On est dans le fichier : Data_20_dom.csv
On est dans le fichier : Data_20_non_dom.csv
On est dans le fichier : Data_20_X.xlsx
On est dans le fichier : Data_21_dom.csv
On est dans le fichier : Data_21_non_dom.csv
On est dans le fichier : Data_21_X.xlsx
On est dans le fichier : Data_22_dom.csv
On est dans le fichier : Data_22_no

In [ ]:
#Filtre passe bas frequence 17Hz "The signal from the sensors were amplified and low-pass filtered (cutoff frequency:17 Hz) to remove any electronic noise and artifacts."




In [22]:
# Paramètres
sampling_rate = 128 
min_peak_value = 10 
merge_window = int(1 * sampling_rate)  # 1 seconde
gap_threshold = int(0.5 * sampling_rate)  # 0.5 seconde
min_duration = int(1.5 * sampling_rate)  # 1.5 seconde

# 1. Extraire les 3 axes gyroscopiques
gyro_x = X_gyro_dom
gyro_y = Y_gyro_dom
gyro_z = Z_gyro_dom

# 2. Détection des pics > 10°/s sur chaque axe
def get_axis_peaks(axis_data):
    axis_data = np.array(axis_data)
    return np.abs(axis_data[np.abs(axis_data) > min_peak_value])

peaks_x = get_axis_peaks(gyro_x)
peaks_y = get_axis_peaks(gyro_y)
peaks_z = get_axis_peaks(gyro_z)

# 3. Moyenne des pics par axe
mean_x = peaks_x.mean() if len(peaks_x) > 0 else np.inf
mean_y = peaks_y.mean() if len(peaks_y) > 0 else np.inf
mean_z = peaks_z.mean() if len(peaks_z) > 0 else np.inf

# 4. Seuil adaptatif = min des moyennes
adaptive_threshold = min(mean_x, mean_y, mean_z)
print(f"Seuil adaptatif : {adaptive_threshold:.2f} °/s")

# 5. Détection brute : un mouvement si un axe dépasse le seuil
movement_raw = (
    (np.abs(gyro_x) > adaptive_threshold) |
    (np.abs(gyro_y) > adaptive_threshold) |
    (np.abs(gyro_z) > adaptive_threshold)
).astype(int)

print(movement_raw)
"""
# 6. Filtre max mobile (fusionner les mouvements séparés de < 0.5s)
movement_merged = maximum_filter1d(movement_raw, size=merge_window)

# 7. Filtre médian mobile (supprimer les mouvements < 1.5s)
movement_final = medfilt(movement_merged, kernel_size=min_duration | 1)  # kernel must be odd
"""

"""
# 8. Visualisation
plt.figure(figsize=(12, 4))
plt.plot(np.linalg.norm(X_clean[:, :3], axis=1), label="Norme Gyro")
plt.plot(movement_final * adaptive_threshold, label="Mouvement détecté", color='orange')
plt.axhline(adaptive_threshold, color='r', linestyle='--', label="Seuil adaptatif")
plt.legend()
plt.title("Détection de mouvement du bras (seuil adaptatif)")
plt.xlabel("Échantillons")
plt.ylabel("Vitesse angulaire (°/s)")
plt.tight_layout()
plt.show()
"""

Seuil adaptatif : 47.84 °/s
[0 0 0 ... 0 0 0]


'\n# 8. Visualisation\nplt.figure(figsize=(12, 4))\nplt.plot(np.linalg.norm(X_clean[:, :3], axis=1), label="Norme Gyro")\nplt.plot(movement_final * adaptive_threshold, label="Mouvement détecté", color=\'orange\')\nplt.axhline(adaptive_threshold, color=\'r\', linestyle=\'--\', label="Seuil adaptatif")\nplt.legend()\nplt.title("Détection de mouvement du bras (seuil adaptatif)")\nplt.xlabel("Échantillons")\nplt.ylabel("Vitesse angulaire (°/s)")\nplt.tight_layout()\nplt.show()\n'